<a href="https://colab.research.google.com/github/gawlowskiandrzej/Eksploracja_danych_projekt---detekcja-spamu/blob/main/spam_dataset_quality_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spam Dataset Quality Analysis Notebook (Universal)

This notebook evaluates dataset quality using:
- Class separability (PCA)
- Lexical overlap (Jaccard similarity)
- Feature leakage (Chi-square)
- Entropy analysis
- Duplicate detection
- Baseline model sanity check


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.feature_selection import chi2
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from collections import Counter
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter


In [47]:
kaggle_file = "spam_email_dataset.csv"
local_file = "llm_spam_email_dataset.csv"

data_dir = "data"
kaggle_path = os.path.join(data_dir, kaggle_file)
local_path = os.path.join(data_dir, local_file)

os.makedirs(data_dir, exist_ok=True)

def load_from_kaggle():
    return kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        "ssssws/spam-email-detection-dataset-clean-and-ml-ready",
        kaggle_file,
    )

def load_from_csv(path):
    return pd.read_csv(path)

if os.path.exists(kaggle_path):
    print("Kaggle CSV już istnieje lokalnie")
    df_kaggle = load_from_csv(kaggle_path)
else:
    print("Pobieram dane z Kaggle...")
    df_kaggle = load_from_kaggle()
    df_kaggle.to_csv(kaggle_path, index=False)

if os.path.exists(local_path):
    print("Wczytuję lokalny dataset (LLM)...")
    df_local = load_from_csv(local_path)
else:
    print(f"Brak pliku {local_file} w folderze data!")

Kaggle CSV już istnieje lokalnie
Wczytuję lokalny dataset (LLM)...


In [48]:
TEXT_COL = 'email_text'
LABEL_COL = 'label'

In [49]:
def compute_jaccard(df):
    spam_words = set(' '.join(df[df[LABEL_COL] == 1][TEXT_COL]).lower().split())
    ham_words = set(' '.join(df[df[LABEL_COL] == 0][TEXT_COL]).lower().split())

    jaccard = len(spam_words & ham_words) / len(spam_words | ham_words)

    if jaccard < 0.2:
        conclusion = "🔴 very low overlap (easy / keyword-driven)"
    elif jaccard < 0.5:
        conclusion = "🟡 moderate overlap (realistic)"
    else:
        conclusion = "🟢 high overlap (hard / semantic)"

    return jaccard, conclusion, len(spam_words), len(ham_words)

k_jaccard, k_conclusion, k_spam, k_ham = compute_jaccard(df_kaggle)
l_jaccard, l_conclusion, l_spam, l_ham = compute_jaccard(df_local)

comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "Jaccard Similarity": [k_jaccard, l_jaccard],
    "Spam vocab size": [k_spam, l_spam],
    "Ham vocab size": [k_ham, l_ham],
    "Conclusion": [k_conclusion, l_conclusion]
})

comparison_df

,Dataset,Jaccard Similarity,Spam vocab size,Ham vocab size,Conclusion
0,Kaggle,0.988730,1935,1947,🟢 high overlap (hard / semantic)
1,Local (LLM),0.210151,743,783,🟡 moderate overlap (realistic)


In [50]:
KEYWORDS = ['free','win','click','urgent','offer']

def chi2_analysis(df):
    cv = CountVectorizer(stop_words='english', min_df=5)
    X_counts = cv.fit_transform(df[TEXT_COL])

    chi_scores, _ = chi2(X_counts, df[LABEL_COL])
    top_idx = chi_scores.argsort()[-20:]

    features = np.array(cv.get_feature_names_out())[top_idx]

    keyword_hits = sum(f in KEYWORDS for f in features)

    if keyword_hits >= 10:
        conclusion = "🔴 strong keyword reliance (too easy)"
    elif keyword_hits >= 5:
        conclusion = "🟡 partial keyword patterns"
    else:
        conclusion = "🟢 low keyword leakage (semantic)"

    return features, keyword_hits, conclusion

k_features, k_hits, k_conclusion = chi2_analysis(df_kaggle)
l_features, l_hits, l_conclusion = chi2_analysis(df_local)


comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "Keyword hits (top20)": [k_hits, l_hits],
    "Conclusion": [k_conclusion, l_conclusion],
    "Top features": [', '.join(k_features), ', '.join(l_features)]
})

comparison_df

,Dataset,Keyword hits (top20),Conclusion,Top features
0,Kaggle,5,🟡 partial keyword patterns,"sense, peace, help, budget, meeting, project, ..."
1,Local (LLM),1,🟢 low keyword leakage (semantic),"days, did, 24, details, subscription, know, le..."


In [51]:
def entropy(texts):
    words = ' '.join(texts).lower().split()
    freq = Counter(words)
    probs = np.array(list(freq.values())) / sum(freq.values())
    return -np.sum(probs * np.log2(probs))

def entropy_analysis(df):
    spam_ent = entropy(df[df[LABEL_COL] == 1][TEXT_COL])
    ham_ent = entropy(df[df[LABEL_COL] == 0][TEXT_COL])

    diff = abs(spam_ent - ham_ent)

    if diff < 0.2:
        conclusion = "🟢 balanced linguistic complexity"
    elif spam_ent > ham_ent:
        conclusion = "🟡 spam more diverse"
    else:
        conclusion = "🔴 entropy imbalance (bias/artifacts)"

    return spam_ent, ham_ent, diff, conclusion

k_spam_ent, k_ham_ent, k_diff, k_conclusion = entropy_analysis(df_kaggle)
l_spam_ent, l_ham_ent, l_diff, l_conclusion = entropy_analysis(df_local)

comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "Spam entropy": [k_spam_ent, l_spam_ent],
    "Ham entropy": [k_ham_ent, l_ham_ent],
    "Abs diff": [k_diff, l_diff],
    "Conclusion": [k_conclusion, l_conclusion]
})

comparison_df

,Dataset,Spam entropy,Ham entropy,Abs diff,Conclusion
0,Kaggle,7.456187,10.007867,2.551680,🔴 entropy imbalance (bias/artifacts)
1,Local (LLM),8.376785,8.489771,0.112986,🟢 balanced linguistic complexity


In [52]:
def length_analysis(df):
    df = df.copy()
    df['length'] = df[TEXT_COL].apply(len)

    stats = df.groupby(LABEL_COL)['length'].describe()

    spam_mean = df[df[LABEL_COL] == 1]['length'].mean()
    ham_mean = df[df[LABEL_COL] == 0]['length'].mean()

    ratio = spam_mean / (ham_mean + 1e-6)

    if 0.8 <= ratio <= 1.2:
        conclusion = "🟢 similar lengths (realistic)"
    elif ratio > 1.5:
        conclusion = "🟡 spam much longer (bias)"
    else:
        conclusion = "🔴 strong length imbalance"

    return spam_mean, ham_mean, ratio, conclusion, stats

k_spam_mean, k_ham_mean, k_ratio, k_conclusion, k_stats = length_analysis(df_kaggle)
l_spam_mean, l_ham_mean, l_ratio, l_conclusion, l_stats = length_analysis(df_local)

comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "Spam mean length": [k_spam_mean, l_spam_mean],
    "Ham mean length": [k_ham_mean, l_ham_mean],
    "Spam/Ham ratio": [k_ratio, l_ratio],
    "Conclusion": [k_conclusion, l_conclusion]
})

comparison_df

,Dataset,Spam mean length,Ham mean length,Spam/Ham ratio,Conclusion
0,Kaggle,131.297622,104.278102,1.259110,🔴 strong length imbalance
1,Local (LLM),226.479167,202.538462,1.118203,🟢 similar lengths (realistic)


In [53]:
def similarity_analysis(df, sample_size=2000):
    df = df.copy()

    if len(df) > sample_size:
        df = df.sample(sample_size, random_state=42)

    tfidf = TfidfVectorizer().fit_transform(df[TEXT_COL])

    sim = cosine_similarity(tfidf)
    np.fill_diagonal(sim, 0)

    max_sim = sim.max()

    if max_sim > 0.9:
        conclusion = "🔴 very high duplication (low quality / synthetic)"
    elif max_sim > 0.75:
        conclusion = "🟡 moderate duplication"
    else:
        conclusion = "🟢 low duplication (good diversity)"

    return max_sim, conclusion, len(df)

# --- obliczenia ---
k_sim, k_conclusion, k_n = similarity_analysis(df_kaggle)
l_sim, l_conclusion, l_n = similarity_analysis(df_local)

# --- tabela ---
comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "Max cosine similarity": [k_sim, l_sim],
    "Sample size": [k_n, l_n],
    "Conclusion": [k_conclusion, l_conclusion]
})

comparison_df

,Dataset,Max cosine similarity,Sample size,Conclusion
0,Kaggle,0.591229,2000,🟢 low duplication (good diversity)
1,Local (LLM),0.653389,100,🟢 low duplication (good diversity)


In [54]:
def cv_analysis(df):
    vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
    X = vectorizer.fit_transform(df[TEXT_COL])

    model = LogisticRegression(max_iter=1000)
    scores = cross_val_score(model, X, df[LABEL_COL], cv=5)

    acc = scores.mean()

    if acc > 0.95:
        conclusion = "🔴 likely leakage / too easy dataset"
    elif acc > 0.85:
        conclusion = "🟡 learnable patterns but some structure"
    else:
        conclusion = "🟢 harder / more semantic reasoning required"

    return acc, conclusion

# --- obliczenia ---
k_acc, k_conclusion = cv_analysis(df_kaggle)
l_acc, l_conclusion = cv_analysis(df_local)

# --- tabela porównawcza ---
comparison_df = pd.DataFrame({
    "Dataset": ["Kaggle", "Local (LLM)"],
    "CV Accuracy": [k_acc, l_acc],
    "Conclusion": [k_conclusion, l_conclusion]
})

comparison_df

,Dataset,CV Accuracy,Conclusion
0,Kaggle,1.00,🔴 likely leakage / too easy dataset
1,Local (LLM),0.61,🟢 harder / more semantic reasoning required
